# PCA + KMeans by group

按照 `PCA_Harmony_Kmeans.ipynb` 后半段标准曲线计算思路：先对 6 个 group (`ctrl1`, `ctrl2`, `ctrl3`, `dn1`, `dn2`, `dn3`) 分别 PCA + KMeans，得到 cluster 信息后按 `group + cluster` 计算平均 intensity，再拟合标准曲线。

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans

In [2]:
output_path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_PCA_Kmeans"
os.makedirs(output_path, exist_ok=True)

adata_path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Leiden2/adata30.h5ad"
ctrl_intensity_path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_PCA_Harmony_Leiden/ctrlIntensity_merged.csv"
dn_intensity_path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_PCA_Harmony_Leiden/dnIntensity_merged.csv"

In [3]:
adata = sc.read_h5ad(adata_path)

In [4]:
adata.obs

,sample,x,y
ctrl1_0_pixel1,ctrl1_0,61072.238281,10187.917969
ctrl1_0_pixel2,ctrl1_0,61092.238281,10187.917969
ctrl1_0_pixel3,ctrl1_0,61112.238281,10187.917969
ctrl1_0_pixel4,ctrl1_0,60572.238281,10167.917969
ctrl1_0_pixel5,ctrl1_0,60592.238281,10167.917969
...,...,...,...
dn3_60_pixel58565,dn3_60,2359.611816,5964.661621
dn3_60_pixel58566,dn3_60,2379.611816,5964.661621
dn3_60_pixel58567,dn3_60,2399.611816,5964.661621
dn3_60_pixel58568,dn3_60,2419.611816,5964.661621


In [5]:

adata.obs["library_id"] = adata.obs["sample"].astype(str)
parts = adata.obs["sample"].astype(str).str.extract(r"^([A-Za-z]+)(\d+)_")
adata.obs["condition"] = parts[0].astype(str)
adata.obs["replicate"] = "Rep" + parts[1].astype(str)
adata.obs["pca_kmeans_group"] = adata.obs["sample"].astype(str).str.extract(r"^([A-Za-z]+\d+)_")[0]

expected_groups = ["ctrl1", "ctrl2", "ctrl3", "dn1", "dn2", "dn3"]
groups = [g for g in expected_groups if g in set(adata.obs["pca_kmeans_group"].astype(str))]
missing_groups = sorted(set(expected_groups) - set(groups))
print("groups:", groups)
if missing_groups:
    print("missing groups:", missing_groups)

adata.obs[["sample", "library_id", "condition", "replicate", "pca_kmeans_group"]].head()

groups: ['ctrl1', 'ctrl2', 'ctrl3', 'dn1', 'dn2', 'dn3']


,sample,library_id,condition,replicate,pca_kmeans_group
ctrl1_0_pixel1,ctrl1_0,ctrl1_0,ctrl,Rep1,ctrl1
ctrl1_0_pixel2,ctrl1_0,ctrl1_0,ctrl,Rep1,ctrl1
ctrl1_0_pixel3,ctrl1_0,ctrl1_0,ctrl,Rep1,ctrl1
ctrl1_0_pixel4,ctrl1_0,ctrl1_0,ctrl,Rep1,ctrl1
ctrl1_0_pixel5,ctrl1_0,ctrl1_0,ctrl,Rep1,ctrl1


In [6]:
n_comps = 50
n_pcs_use = 30
n_clusters = 19

adata.obs["kmeans"] = pd.Series(index=adata.obs_names, dtype="object")

for group in groups:
    mask = adata.obs["pca_kmeans_group"].astype(str).to_numpy() == group
    adata_group = adata[mask].copy()

    n_comps_group = min(n_comps, adata_group.n_obs - 1, adata_group.n_vars - 1)
    if n_comps_group < 2:
        raise ValueError(f"{group} 可用于 PCA 的维度不足: n_obs={adata_group.n_obs}, n_vars={adata_group.n_vars}")

    sc.pp.pca(adata_group, n_comps=n_comps_group, svd_solver="arpack", random_state=0)
    use_n = min(n_pcs_use, n_comps_group)
    X_kmeans = adata_group.obsm["X_pca"][:, :use_n]

    n_clusters_group = min(n_clusters, adata_group.n_obs)
    kmeans = KMeans(
        n_clusters=n_clusters_group,
        init="k-means++",
        n_init="auto",
        random_state=0,
    )
    labels = kmeans.fit_predict(X_kmeans).astype(str)

    # cluster 编号在每个 group 内独立，例如 ctrl1 的 0 和 dn1 的 0 互不混用。
    adata.obs.loc[mask, "kmeans"] = labels
    print(f"{group}: n_obs={adata_group.n_obs}, n_pcs={use_n}, n_clusters={n_clusters_group}")

adata.obs["kmeans"] = adata.obs["kmeans"].astype("category")
adata.obs["cluster"] = adata.obs["kmeans"].astype(str).astype("category")
adata.write(f"{output_path}/adata30_after_group_pca_kmeans.h5ad")
adata.obs[["sample", "pca_kmeans_group", "cluster"]].head()

ctrl1: n_obs=154643, n_pcs=30, n_clusters=19
ctrl2: n_obs=95832, n_pcs=30, n_clusters=19
ctrl3: n_obs=247361, n_pcs=30, n_clusters=19
dn1: n_obs=143727, n_pcs=30, n_clusters=19
dn2: n_obs=155635, n_pcs=30, n_clusters=19
dn3: n_obs=306211, n_pcs=30, n_clusters=19


,sample,pca_kmeans_group,cluster
ctrl1_0_pixel1,ctrl1_0,ctrl1,6
ctrl1_0_pixel2,ctrl1_0,ctrl1,6
ctrl1_0_pixel3,ctrl1_0,ctrl1,6
ctrl1_0_pixel4,ctrl1_0,ctrl1,6
ctrl1_0_pixel5,ctrl1_0,ctrl1,6


In [ ]:
# 用全体数据的 PCA 表达计算邻居和 UMAP，仅用于可视化；cluster 仍来自各 group 内独立 PCA+KMeans。
global_n_comps = min(n_comps, adata.n_obs - 1, adata.n_vars - 1)
sc.pp.pca(adata, n_comps=global_n_comps, svd_solver="arpack", random_state=0)
sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=min(n_pcs_use, global_n_comps),
    metric="euclidean",
    random_state=0,
)
sc.tl.umap(adata, random_state=0)
adata.write(f"{output_path}/adata30_group_pca_kmeans_umap.h5ad")

In [ ]:
sc.pl.umap(
    adata,
    color=["cluster", "condition", "replicate", "pca_kmeans_group"],
    wspace=0.35,
    legend_loc="right margin",
    size=8,
    show=False,
)
plt.savefig(
    os.path.join(output_path, "umap_group_pca_kmeans.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.close()

cats = adata.obs["cluster"].cat.categories.tolist()
cmap = plt.get_cmap("tab20", len(cats))
palette = {c: mpl.colors.to_hex(cmap(i)) for i, c in enumerate(cats)}
handles = [
    Line2D([0], [0], marker="o", color="w", label=c, markerfacecolor=palette[c], markersize=6)
    for c in cats
]

library_ids = sorted(adata.obs["library_id"].astype(str).unique().tolist())
ncols = 5
nrows = int(np.ceil(len(library_ids) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)
umap = adata.obsm["X_umap"]

for ax, lib in zip(axes, library_ids):
    idx = adata.obs["library_id"].astype(str).to_numpy() == lib
    colors = adata.obs.loc[idx, "cluster"].map(palette).values
    ax.scatter(umap[idx, 0], umap[idx, 1], c=colors, s=3, linewidths=0)
    ax.set_title(lib)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")

for ax in axes[len(library_ids):]:
    ax.axis("off")

fig.legend(handles=handles, labels=cats, loc="center right", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(rect=[0, 0, 0.94, 1])
plt.savefig(os.path.join(output_path, "umap_by_library_group_pca_kmeans.png"), dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, lib in zip(axes, library_ids):
    idx = adata.obs["library_id"].astype(str).to_numpy() == lib
    colors = adata.obs.loc[idx, "cluster"].map(palette).values
    ax.scatter(
        adata.obs.loc[idx, "x"].astype(float),
        adata.obs.loc[idx, "y"].astype(float),
        c=colors,
        s=3,
        linewidths=0,
    )
    ax.invert_yaxis()
    ax.set_title(lib)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

for ax in axes[len(library_ids):]:
    ax.axis("off")

fig.legend(handles=handles, labels=cats, loc="center right", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(rect=[0, 0, 0.94, 1])
plt.savefig(os.path.join(output_path, "spatial_by_library_group_pca_kmeans.png"), dpi=300, bbox_inches="tight")
plt.close()

slice_dir = os.path.join(output_path, "spatial_by_slice")
os.makedirs(slice_dir, exist_ok=True)

for lib in library_ids:
    idx = adata.obs["library_id"].astype(str).to_numpy() == lib
    colors = adata.obs.loc[idx, "cluster"].map(palette).values

    plt.figure(figsize=(5, 5))
    plt.scatter(
        adata.obs.loc[idx, "x"].astype(float),
        adata.obs.loc[idx, "y"].astype(float),
        c=colors,
        s=3,
        linewidths=0,
    )
    plt.gca().invert_yaxis()
    plt.title(lib)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.tight_layout()
    plt.savefig(os.path.join(slice_dir, f"{lib}_spatial_group_pca_kmeans.png"), dpi=300, bbox_inches="tight")
    plt.close()

adata.obs.to_csv(f"{output_path}/final_obs_metadata_group_pca_kmeans.csv")

In [7]:
ctrl_merged = pd.read_csv(ctrl_intensity_path, sep=";", index_col=0)
dn_merged = pd.read_csv(dn_intensity_path, sep=";", index_col=0)
print(ctrl_merged.shape)
print(dn_merged.shape)

(497836, 1245)
(605573, 1120)


In [8]:
ctrl_cluster = adata.obs.loc[
    adata.obs["library_id"].astype(str).str.startswith("ctrl"),
    "cluster",
].reset_index(drop=True)

dn_cluster = adata.obs.loc[
    adata.obs["library_id"].astype(str).str.startswith("dn"),
    "cluster",
].reset_index(drop=True)

if len(ctrl_cluster) != len(ctrl_merged):
    raise ValueError(
        f"ctrl 的 cluster 数量 ({len(ctrl_cluster)}) 与 ctrl_merged 行数 ({len(ctrl_merged)}) 不一致"
    )
if len(dn_cluster) != len(dn_merged):
    raise ValueError(
        f"dn 的 cluster 数量 ({len(dn_cluster)}) 与 dn_merged 行数 ({len(dn_merged)}) 不一致"
    )

ctrl_merged = ctrl_merged.copy()
dn_merged = dn_merged.copy()
ctrl_merged.insert(0, "cluster", ctrl_cluster.to_numpy())
dn_merged.insert(0, "cluster", dn_cluster.to_numpy())
print(ctrl_merged.iloc[:5, :5])
print(dn_merged.iloc[:5, :5])

               cluster   57.0346   58.0302   59.0139   67.0191
ctrl1_0_pixel1       6  0.000000  0.000000  0.261729  0.000000
ctrl1_0_pixel2       6  0.000000  0.000000  0.140146  0.414068
ctrl1_0_pixel3       6  0.000000  0.258983  0.000000  0.000000
ctrl1_0_pixel4       6  0.201059  0.000000  0.000000  0.495467
ctrl1_0_pixel5       6  0.577206  0.000000  0.849775  0.000000
             cluster   57.0346   58.0302   59.0139   67.0191
dn1_0_pixel1       1  0.000000  0.712215  0.000000  0.000000
dn1_0_pixel2       1  0.000000  1.836354  3.292773  1.308666
dn1_0_pixel3       1  0.000000  1.547361  0.000000  1.246902
dn1_0_pixel4       1  0.830629  0.540409  0.000000  0.000000
dn1_0_pixel5       1  0.000000  0.000000  0.000000  0.202554


In [9]:
def build_cluster_mean(intensity_df, prefix):
    mz_cols = [c for c in intensity_df.columns if c != "cluster"]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))
    intensity_df = intensity_df[["cluster"] + mz_cols].copy()

    tmp = intensity_df.copy()
    tmp["group"] = tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)
    cluster_mean = tmp.groupby(["group", "cluster"], sort=False)[mz_cols].mean()
    cluster_mean.index = [
        f"{group}_cluster{cluster}"
        for group, cluster in cluster_mean.index
    ]

    sort_df = cluster_mean.copy()
    parts = sort_df.index.to_series().str.extract(
        rf"^{prefix}(\d+)_(\d+)_cluster(\d+)$"
    )
    sort_df["_sample_num"] = parts[0].astype(int)
    sort_df["_time"] = parts[1].astype(int)
    sort_df["_cluster"] = parts[2].astype(int)
    sort_df = sort_df.sort_values(
        by=["_sample_num", "_cluster", "_time"],
        kind="stable",
    )
    return sort_df.drop(columns=["_sample_num", "_time", "_cluster"])

ctrl_cluster_mean = build_cluster_mean(ctrl_merged, "ctrl")
dn_cluster_mean = build_cluster_mean(dn_merged, "dn")

ctrl_cluster_mean.to_csv(f"{output_path}/ctrlIntensityMeanByCluster.csv")
dn_cluster_mean.to_csv(f"{output_path}/dnIntensityMeanByCluster.csv")

print(ctrl_cluster_mean.iloc[:10, :5])
print(dn_cluster_mean.iloc[:10, :5])

                    57.0346   58.0302   59.0139   67.0191   68.0143
ctrl1_0_cluster0   0.636711  1.115291  4.015964  1.101595  2.253472
ctrl1_15_cluster0  0.700977  1.187865  4.488714  1.162658  1.995130
ctrl1_30_cluster0  0.733013  1.100446  4.780336  1.237051  1.600146
ctrl1_45_cluster0  0.771449  1.132815  5.174461  1.296789  1.617152
ctrl1_60_cluster0  0.838079  1.131846  5.499151  1.389598  1.601668
ctrl1_0_cluster1   0.615959  1.128640  3.879701  1.091920  1.959518
ctrl1_15_cluster1  0.729841  1.228064  4.750989  1.199292  1.908958
ctrl1_30_cluster1  0.754394  1.093915  5.176024  1.311586  1.711891
ctrl1_45_cluster1  0.788921  1.096567  5.385602  1.297348  1.782443
ctrl1_60_cluster1  0.788774  1.143877  5.782102  1.392715  1.630891
                  57.0346   58.0302   59.0139   67.0191   68.0143
dn1_0_cluster0   0.688794  0.884190  4.770709  1.098377  2.076136
dn1_15_cluster0  0.734043  1.039516  4.879510  1.333889  1.759141
dn1_30_cluster0  0.830141  1.193806  5.348035  1.43482

In [11]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs


def build_linreg_result(df):
    tmp = df.copy()
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )

    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)

    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))

    result_rows = []
    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats(x, y)
            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs,
            })

    result_df = pd.DataFrame(result_rows)
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)
    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable",
    ).drop(columns=["_prefix", "_sample_num"])
    return result_df.reset_index(drop=True)

In [12]:
ctrl_linreg_result = build_linreg_result(ctrl_cluster_mean)
dn_linreg_result = build_linreg_result(dn_cluster_mean)

ctrl_linreg_result.to_csv(f"{output_path}/ctrl_linreg_result.csv", index=False)
dn_linreg_result.to_csv(f"{output_path}/dn_linreg_result.csv", index=False)

print(ctrl_linreg_result.head())
print(dn_linreg_result.head())

  sample       mz  cluster     slope  intercept        r2       x0_abs
0  ctrl1  57.0346        0  0.003155   0.641404  0.983400   203.316165
1  ctrl1  58.0302        0 -0.000146   1.138040  0.010982  7780.825367
2  ctrl1  59.0139        0  0.024347   4.061301  0.994861   166.805889
3  ctrl1  67.0191        0  0.004734   1.095511  0.994259   231.400934
4  ctrl1  68.0143        0 -0.011211   2.149831  0.795401   191.768278
  sample       mz  cluster     slope  intercept        r2      x0_abs
0    dn1  57.0346        0  0.002140   0.712216  0.664311  332.881077
1    dn1  58.0302        0  0.003122   0.969268  0.414339  310.488282
2    dn1  59.0139        0  0.016101   4.764223  0.897701  295.894779
3    dn1  67.0191        0  0.006487   1.177739  0.845706  181.565488
4    dn1  68.0143        0 -0.006623   1.994337  0.759295  301.114059


In [13]:
ctrl_r2_mean = (
    ctrl_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)
ctrl_good = ctrl_r2_mean[ctrl_r2_mean["mean_r2"] >= 0.9].copy()
ctrl_count = (
    ctrl_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

dn_r2_mean = (
    dn_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)
dn_good = dn_r2_mean[dn_r2_mean["mean_r2"] >= 0.9].copy()
dn_count = (
    dn_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

ctrl_good.to_csv(f"{output_path}/ctrl_mz_mean_r2_ge_0.9.csv", index=False)
ctrl_count.to_csv(f"{output_path}/ctrl_mz_mean_r2_ge_0.9_count.csv", index=False)
dn_good.to_csv(f"{output_path}/dn_mz_mean_r2_ge_0.9.csv", index=False)
dn_count.to_csv(f"{output_path}/dn_mz_mean_r2_ge_0.9_count.csv", index=False)

print("ctrl_good:")
print(ctrl_good.head())
print("ctrl_count:")
print(ctrl_count)
print("dn_good:")
print(dn_good.head())
print("dn_count:")
print(dn_count)

ctrl_good:
   sample       mz   mean_r2
2   ctrl1  59.0139  0.944646
3   ctrl1  67.0191  0.922058
9   ctrl1  71.0140  0.912928
10  ctrl1  71.0503  0.911474
14  ctrl1  73.0294  0.927409
ctrl_count:
  sample  n_metabolites_r2_ge_0.9
0  ctrl1                      225
1  ctrl2                      135
2  ctrl3                      370
dn_good:
   sample        mz   mean_r2
34    dn1   88.0405  0.933032
35    dn1   89.0246  0.948324
46    dn1   96.9696  0.930715
52    dn1   98.9740  0.907784
56    dn1  100.0042  0.914001
dn_count:
  sample  n_metabolites_r2_ge_0.9
0    dn1                      110
1    dn2                      193
2    dn3                      189


In [ ]:
def mz_first_two_before_decimal(mz):
    integer_part = str(int(float(mz)))
    return integer_part[:2]

all_linreg_result = pd.concat(
    [
        ctrl_linreg_result.assign(condition="ctrl"),
        dn_linreg_result.assign(condition="dn"),
    ],
    ignore_index=True,
)
all_linreg_result["mz_key"] = all_linreg_result["mz"].map(mz_first_two_before_decimal)

expected_samples = [g for g in expected_groups if g in set(all_linreg_result["sample"].astype(str))]
expected_pairs = (
    all_linreg_result.loc[all_linreg_result["sample"].isin(expected_samples), ["sample", "cluster"]]
    .drop_duplicates()
    .sort_values(["sample", "cluster"], kind="stable")
)
expected_pair_count = len(expected_pairs)

mz_key_pair_best = (
    all_linreg_result.loc[all_linreg_result["sample"].isin(expected_samples)]
    .groupby(["mz_key", "sample", "cluster"], as_index=False)
    .agg(
        max_r2=("r2", "max"),
        representative_mz=("mz", "first"),
        n_mz_in_key=("mz", "nunique"),
    )
)

mz_key_summary = (
    mz_key_pair_best
    .groupby("mz_key", as_index=False)
    .agg(
        n_sample_cluster_pairs=("max_r2", "size"),
        min_r2_across_all_clusters=("max_r2", "min"),
        mean_r2_across_all_clusters=("max_r2", "mean"),
        representative_mz=("representative_mz", "first"),
        n_mz_in_key=("n_mz_in_key", "max"),
    )
)

mz_keys_r2_ge_0_9_all_clusters = mz_key_summary[
    (mz_key_summary["n_sample_cluster_pairs"] == expected_pair_count)
    & (mz_key_summary["min_r2_across_all_clusters"] >= 0.9)
].copy()

mz_keys_r2_ge_0_9_all_clusters = mz_keys_r2_ge_0_9_all_clusters.sort_values(
    ["mz_key"],
    key=lambda s: s.astype(int),
    kind="stable",
).reset_index(drop=True)

mz_key_pair_best.to_csv(f"{output_path}/all_groups_cluster_r2_by_mz_key.csv", index=False)
mz_key_summary.to_csv(f"{output_path}/all_groups_cluster_r2_mz_key_summary.csv", index=False)
mz_keys_r2_ge_0_9_all_clusters.to_csv(
    f"{output_path}/mz_key_r2_ge_0.9_in_all_6_groups_all_clusters.csv",
    index=False,
)

print("expected samples:", expected_samples)
print("sample+cluster pairs:", expected_pair_count)
print("mz_key count with R2 >= 0.9 in all 6 groups/all clusters:", len(mz_keys_r2_ge_0_9_all_clusters))
print(mz_keys_r2_ge_0_9_all_clusters)
